# Massive SPX/VIX/SPY Database — Starter

This notebook validates access, downloads underlyings, builds Parquet and DuckDB storage, discovers SPX option identifiers, saves option metadata, collects current option-chain snapshots, and demonstrates a controlled historical option sample.

**The API key is loaded from `.env` and is never displayed.**

In [17]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
import os

PROJECT_ROOT = Path.cwd()
ENV_PATH = PROJECT_ROOT / "main.env"

print("Notebook working directory:", PROJECT_ROOT)
print("Expected environment file:", ENV_PATH)
print("File exists:", ENV_PATH.is_file())

if not ENV_PATH.is_file():
    print("Files containing 'env' in the project directory:")
    print([path.name for path in PROJECT_ROOT.glob("*env*")])
    raise FileNotFoundError(f"Environment file not found: {ENV_PATH}")

# Parse the file without putting its values into os.environ.
parsed = dotenv_values(ENV_PATH)

print("Parsed variable names:", list(parsed.keys()))
print("MASSIVE_API_KEY found:", "MASSIVE_API_KEY" in parsed)
print("MASSIVE_API_KEY has a value:", bool(parsed.get("MASSIVE_API_KEY")))

# override=True avoids a stale or empty variable in the notebook kernel
loaded = load_dotenv(ENV_PATH, override=True, verbose=True)

API_KEY = os.getenv("MASSIVE_API_KEY")

print("load_dotenv loaded variables:", loaded)
print("API key available:", bool(API_KEY))
print("API key length:", len(API_KEY) if API_KEY else 0)

Notebook working directory: c:\Users\hkhat\OneDrive\Desktop\Project\Dissertation
Expected environment file: c:\Users\hkhat\OneDrive\Desktop\Project\Dissertation\main.env
File exists: True
Parsed variable names: ['MASSIVE_API_KEY']
MASSIVE_API_KEY found: True
MASSIVE_API_KEY has a value: True
load_dotenv loaded variables: True
API key available: True
API key length: 32


In [19]:
from pathlib import Path
import os
import sys
from datetime import date, timedelta

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError("Run this notebook from the massive_market_database folder.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from massive_database import (
    MassiveREST,
    data_quality_report,
    discover_option_underlying,
    download_aggregate_history,
    download_selected_option_bars,
    initialize_duckdb,
    latest_regular_session_price,
    normalize_aggregates,
    normalize_option_chain,
    normalize_option_contracts,
    register_parquet_views,
    select_near_atm_contracts,
    write_snapshot_parquet,
)

DATA_ROOT = PROJECT_ROOT / "data"
DB_PATH = DATA_ROOT / "market.duckdb"
DATA_ROOT.mkdir(exist_ok=True)

load_dotenv(PROJECT_ROOT / "main.env")
API_KEY = os.getenv("MASSIVE_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "MASSIVE_API_KEY is missing from the environment. Please add it to main.env or set it in your environment variables."
    )

client = MassiveREST(API_KEY)
con = initialize_duckdb(DB_PATH)
print("Configuration loaded. The API key is present but hidden.")
print("DuckDB:", DB_PATH)

Configuration loaded. The API key is present but hidden.
DuckDB: c:\Users\hkhat\OneDrive\Desktop\Project\Dissertation\data\market.duckdb


## 1. Validate underlying access

This searches recent weekdays rather than assuming yesterday was a trading session.

In [20]:
def recent_data_sample(ticker: str, asset_class: str, lookback_days: int = 14):
    today = date.today()
    for days_back in range(1, lookback_days + 1):
        candidate = (today - timedelta(days=days_back)).isoformat()
        try:
            raw = client.aggregates(ticker, candidate, candidate)
            if raw:
                return candidate, normalize_aggregates(raw, ticker, asset_class)
        except Exception as exc:
            return None, pd.DataFrame({"ticker": [ticker], "error": [str(exc)]})
    return None, pd.DataFrame()

checks, samples = [], {}
for ticker, asset_class in [("SPY", "stock"), ("I:SPX", "index"), ("I:VIX", "index")]:
    sample_date, frame = recent_data_sample(ticker, asset_class)
    samples[ticker] = frame
    checks.append({
        "ticker": ticker,
        "asset_class": asset_class,
        "sample_date": sample_date,
        "rows": len(frame),
        "status": "ok" if sample_date else "no data or no access",
    })

access_report = pd.DataFrame(checks)
display(access_report)

,ticker,asset_class,sample_date,rows,status
0,SPY,stock,2026-07-17,920,ok
1,I:SPX,index,2026-07-17,396,ok
2,I:VIX,index,2026-07-17,761,ok


In [21]:
spx_sample = samples.get("I:SPX", pd.DataFrame())
display(spx_sample.head())
data_quality_report(spx_sample)

,ticker,asset_class,timestamp_ms,timestamp_utc,timestamp_et,session_date,open,high,low,close,volume,vwap,transactions,otc
0,I:SPX,index,1784295000000,2026-07-17 13:30:00+00:00,2026-07-17 09:30:00,2026-07-17,7447.52,7447.71,7441.62,7447.52,<NA>,<NA>,<NA>,<NA>
1,I:SPX,index,1784295060000,2026-07-17 13:31:00+00:00,2026-07-17 09:31:00,2026-07-17,7447.25,7448.48,7445.36,7447.91,<NA>,<NA>,<NA>,<NA>
2,I:SPX,index,1784295120000,2026-07-17 13:32:00+00:00,2026-07-17 09:32:00,2026-07-17,7447.98,7447.98,7441.51,7442.47,<NA>,<NA>,<NA>,<NA>
3,I:SPX,index,1784295180000,2026-07-17 13:33:00+00:00,2026-07-17 09:33:00,2026-07-17,7442.32,7442.95,7436.53,7436.53,<NA>,<NA>,<NA>,<NA>
4,I:SPX,index,1784295240000,2026-07-17 13:34:00+00:00,2026-07-17 09:34:00,2026-07-17,7436.67,7436.67,7433.30,7433.63,<NA>,<NA>,<NA>,<NA>


{'rows': 396,
 'duplicate_keys': 0,
 'missing_close': 0,
 'first_timestamp_utc': '2026-07-17 13:30:00+00:00',
 'last_timestamp_utc': '2026-07-17 20:05:00+00:00'}

## 2. Proof-of-concept storage

The downloader is incremental. Successfully downloaded sessions are skipped on reruns.

In [22]:
end_date = (date.today() - timedelta(days=1)).isoformat()
start_date = (date.today() - timedelta(days=10)).isoformat()

for ticker, asset_class in [("SPY", "stock"), ("I:SPX", "index"), ("I:VIX", "index")]:
    result = download_aggregate_history(
        client=client,
        con=con,
        data_root=DATA_ROOT,
        ticker=ticker,
        asset_class=asset_class,
        start_date=start_date,
        end_date=end_date,
    )
    print(ticker)
    display(pd.DataFrame([r.__dict__ for r in result]).tail())

print("Views created:", register_parquet_views(con, DATA_ROOT))

SPY


,dataset,ticker,session_date,rows,file_path,status,message
3,stock_minute_1,SPY,2026-07-13,0,None,skipped,Already downloaded
4,stock_minute_1,SPY,2026-07-14,0,None,skipped,Already downloaded
5,stock_minute_1,SPY,2026-07-15,0,None,skipped,Already downloaded
6,stock_minute_1,SPY,2026-07-16,0,None,skipped,Already downloaded
7,stock_minute_1,SPY,2026-07-17,0,None,skipped,Already downloaded


I:SPX


,dataset,ticker,session_date,rows,file_path,status,message
3,index_minute_1,I:SPX,2026-07-13,0,None,skipped,Already downloaded
4,index_minute_1,I:SPX,2026-07-14,0,None,skipped,Already downloaded
5,index_minute_1,I:SPX,2026-07-15,0,None,skipped,Already downloaded
6,index_minute_1,I:SPX,2026-07-16,0,None,skipped,Already downloaded
7,index_minute_1,I:SPX,2026-07-17,0,None,skipped,Already downloaded


I:VIX


,dataset,ticker,session_date,rows,file_path,status,message
3,index_minute_1,I:VIX,2026-07-13,0,None,skipped,Already downloaded
4,index_minute_1,I:VIX,2026-07-14,0,None,skipped,Already downloaded
5,index_minute_1,I:VIX,2026-07-15,0,None,skipped,Already downloaded
6,index_minute_1,I:VIX,2026-07-16,0,None,skipped,Already downloaded
7,index_minute_1,I:VIX,2026-07-17,0,None,skipped,Already downloaded


Views created: ['stock_minute_1', 'index_minute_1', 'underlying_minute_1', 'underlying_regular_session', 'underlying_15min']


In [23]:
display(
    con.execute(
        """
        SELECT ticker, min(timestamp_et) AS first_bar,
               max(timestamp_et) AS last_bar, count(*) AS rows
        FROM underlying_minute_1
        GROUP BY ticker
        ORDER BY ticker
        """
    ).df()
)

display(
    con.execute(
        """
        SELECT ticker, timestamp_et, open, high, low, close, volume
        FROM underlying_15min
        ORDER BY timestamp_utc DESC
        LIMIT 12
        """
    ).df()
)

,ticker,first_bar,last_bar,rows
0,I_SPX,2026-07-08 09:30:00,2026-07-17 16:05:00,3162
1,I_VIX,2026-07-08 03:15:00,2026-07-17 16:15:00,5729
2,SPY,2026-07-08 04:00:00,2026-07-17 19:59:00,7173


,ticker,timestamp_et,open,high,low,close,volume
0,I_SPX,2026-07-17 15:45:00,7460.47,7469.00,7455.46,7457.78,NaN
1,I_VIX,2026-07-17 15:45:00,18.55,18.56,18.30,18.52,NaN
2,SPY,2026-07-17 15:45:00,743.48,744.40,742.92,743.23,6.083588e+06
3,SPY,2026-07-17 15:30:00,743.17,743.62,742.60,743.48,2.708247e+06
4,I_VIX,2026-07-17 15:30:00,18.77,18.80,18.56,18.56,NaN
5,I_SPX,2026-07-17 15:30:00,7456.69,7461.50,7451.91,7460.37,NaN
6,SPY,2026-07-17 15:15:00,742.74,743.28,742.13,743.16,1.672027e+06
7,I_VIX,2026-07-17 15:15:00,18.82,18.98,18.74,18.76,NaN
8,I_SPX,2026-07-17 15:15:00,7452.08,7458.32,7446.27,7456.63,NaN
9,I_VIX,2026-07-17 15:00:00,18.35,18.81,18.35,18.80,NaN


## 3. Full underlying history

The default range is two years. Move the start date earlier only where plan history allows.

In [24]:
RUN_FULL_UNDERLYING_DOWNLOAD = False
FULL_START_DATE = (
    pd.Timestamp.today().normalize() - pd.DateOffset(years=2)
).date().isoformat()
FULL_END_DATE = (date.today() - timedelta(days=1)).isoformat()

if RUN_FULL_UNDERLYING_DOWNLOAD:
    summaries = []
    for ticker, asset_class in [("SPY", "stock"), ("I:SPX", "index"), ("I:VIX", "index")]:
        results = download_aggregate_history(
            client=client,
            con=con,
            data_root=DATA_ROOT,
            ticker=ticker,
            asset_class=asset_class,
            start_date=FULL_START_DATE,
            end_date=FULL_END_DATE,
        )
        summaries.extend(r.__dict__ for r in results)

    register_parquet_views(con, DATA_ROOT)
    display(
        pd.DataFrame(summaries)
        .groupby(["ticker", "status"])
        .size()
        .unstack(fill_value=0)
    )
else:
    print("Disabled. Set RUN_FULL_UNDERLYING_DOWNLOAD = True when ready.")
    print(FULL_START_DATE, "to", FULL_END_DATE)

Disabled. Set RUN_FULL_UNDERLYING_DOWNLOAD = True when ready.
2024-07-18 to 2026-07-17


## 4. Discover and save SPX option contracts

The notebook tests both `I:SPX` and `SPX` instead of assuming the option-underlying identifier.

In [25]:
reference_dates = access_report.loc[
    access_report["ticker"].eq("I:SPX"), "sample_date"
].dropna()
reference_date = reference_dates.iloc[0] if len(reference_dates) else FULL_END_DATE

option_underlying, discovery_report = discover_option_underlying(
    client,
    candidates=("I:SPX", "SPX"),
    as_of=reference_date,
)
display(discovery_report)
print("Selected option underlying:", option_underlying)

,candidate,contracts_returned,status
0,I:SPX,0,ok
1,SPX,10,ok


Selected option underlying: SPX


In [26]:
if option_underlying:
    expiration_lte = (
        pd.Timestamp(reference_date) + pd.Timedelta(days=14)
    ).date().isoformat()

    contracts_raw = client.option_contracts(
        underlying_ticker=option_underlying,
        as_of=reference_date,
        expiration_date_gte=reference_date,
        expiration_date_lte=expiration_lte,
        expired=True,
        limit=1000,
    )
    contracts = normalize_option_contracts(contracts_raw, reference_date)
    display(contracts.head())
    print("Contracts:", len(contracts))

    if not contracts.empty:
        path = write_snapshot_parquet(
            contracts,
            DATA_ROOT,
            "option_contracts",
            reference_date,
            "contracts.parquet",
        )
        print("Saved:", path)
        register_parquet_views(con, DATA_ROOT)
else:
    contracts = pd.DataFrame()
    print("No option underlying was discovered. Review the access report.")

""


Contracts: 0


## 5. Current option-chain snapshot

This is for collection from now onward. It does not backfill historical Greeks or open interest.

In [27]:
COLLECT_CURRENT_CHAIN = False

if COLLECT_CURRENT_CHAIN and option_underlying:
    today = date.today().isoformat()
    expiry_end = (
        pd.Timestamp(today) + pd.Timedelta(days=14)
    ).date().isoformat()

    chain_raw = client.option_chain_snapshot(
        underlying_asset=option_underlying,
        expiration_date_gte=today,
        expiration_date_lte=expiry_end,
        limit=250,
    )
    chain = normalize_option_chain(chain_raw)
    display(chain.head())
    print("Snapshot rows:", len(chain))

    if not chain.empty:
        path = write_snapshot_parquet(
            chain,
            DATA_ROOT,
            "option_chain_snapshots",
            today,
            "spx_chain.parquet",
        )
        print("Saved:", path)
        register_parquet_views(con, DATA_ROOT)
else:
    print("Disabled. Set COLLECT_CURRENT_CHAIN = True after checking plan access.")

Disabled. Set COLLECT_CURRENT_CHAIN = True after checking plan access.


## 6. Controlled historical option-price sample

This uses the latest SPX price at or before 15:00 ET, limits the universe to ±5% of spot, selects no more than 20 contracts, and downloads one session.

In [28]:
RUN_OPTION_SAMPLE = False
register_parquet_views(con, DATA_ROOT)

if RUN_OPTION_SAMPLE and option_underlying and not contracts.empty:
    spot = latest_regular_session_price(
        con,
        "I:SPX",
        reference_date,
        "15:00:00",
    )
    print("SPX reference price:", spot)

    if spot is None:
        raise RuntimeError("No SPX price found for the reference session.")

    selected = select_near_atm_contracts(
        contracts,
        spot,
        max_contracts=20,
        moneyness_pct=0.05,
    )
    display(
        selected[
            [
                "ticker",
                "contract_type",
                "expiration_date",
                "strike_price",
                "distance_to_spot",
            ]
        ]
    )

    report = download_selected_option_bars(
        client,
        con,
        DATA_ROOT,
        selected,
        reference_date,
    )
    display(report)
    register_parquet_views(con, DATA_ROOT)
else:
    print("Disabled. Set RUN_OPTION_SAMPLE = True after reviewing the contract universe.")

Disabled. Set RUN_OPTION_SAMPLE = True after reviewing the contract universe.


## 7. Audit log

Errors and empty sessions remain visible rather than being silently discarded.

In [29]:
display(
    con.execute(
        """
        SELECT dataset, ticker, status,
               count(*) AS sessions, sum(rows) AS rows
        FROM ingestion_log
        GROUP BY dataset, ticker, status
        ORDER BY dataset, ticker, status
        """
    ).df()
)

display(
    con.execute(
        """
        SELECT *
        FROM ingestion_log
        WHERE status = 'error'
        ORDER BY ingested_at_utc DESC
        LIMIT 20
        """
    ).df()
)

,dataset,ticker,status,sessions,rows
0,index_minute_1,I:SPX,ok,8,3162.0
1,index_minute_1,I:VIX,ok,8,5729.0
2,stock_minute_1,SPY,ok,8,7173.0


,dataset,ticker,session_date,rows,file_path,status,message,ingested_at_utc


## Next notebook

After the underlying download is complete, build a canonical session calendar, align SPX/VIX/SPY without look-ahead, freeze the option-universe sampling rule, and create modelling tables separately from raw storage.